# ЛР4 — проверка tool_calls

GPU T4, Internet On. Запускайте ячейки по порядку. Проверяется один инструмент чтения учебного календаря.

In [ ]:
import os, sys, json, shutil, subprocess, time
from pathlib import Path

ART = Path("/kaggle/working/lab4_smoke")
ART.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ)
ENV.pop("PYTHONPATH", None)
ENV.update(CUDA_VISIBLE_DEVICES="0", HF_HOME="/tmp/lab4-hf",
           PYTHONNOUSERSITE="1", PYTHONUNBUFFERED="1",
           OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")
def cmd(args, **kwargs):
    return subprocess.run(args, env=ENV, check=True, **kwargs)

gpu = cmd(["nvidia-smi", "--id=0", "--query-gpu=name", "--format=csv,noheader"],
          capture_output=True, text=True).stdout.strip()
assert "T4" in gpu, gpu
print(gpu)
if not shutil.which("uv"):
    cmd([sys.executable, "-m", "pip", "install", "uv"])
uv = shutil.which("uv")
assert uv
PYTHON = "/tmp/lab4-env/bin/python"
if not Path(PYTHON).exists():
    cmd([uv, "venv", "--python", "3.13", "--seed", "/tmp/lab4-env"])
cmd([uv, "pip", "install", "--python", PYTHON, "--torch-backend=auto",
     "vllm==0.33.0", "torch==2.14.0+cu132", "transformers==5.20.2",
     "huggingface-hub==2.4.0", "tokenizers==0.23.3", "numpy==2.4.6",
     "openai==3.45.0", "PyYAML==6.0.3", "wrapt==2.5.2"])
cmd([uv, "pip", "check", "--python", PYTHON])
cmd([PYTHON, "-c", "import torch; assert torch.cuda.is_available() and torch.cuda.device_count()==1; print(torch.cuda.get_device_name(0))"])
freeze = cmd([uv, "pip", "freeze", "--python", PYTHON], capture_output=True, text=True)
(ART/"requirements_gpu.txt").write_text(freeze.stdout, encoding="utf-8")


In [ ]:
import urllib.request

MODEL = "Qwen/Qwen3-4B-Instruct-2507"
REV = "cdbee75f17c01a7cc42f958dc650907174af0554"
BASE = "http://127.0.0.1:8000"
command = ["/tmp/lab4-env/bin/vllm", "serve", MODEL,
           "--revision", REV, "--tokenizer-revision", REV,
           "--dtype", "half", "--max-model-len", "16384",
           "--max-num-seqs", "1", "--max-num-batched-tokens", "2048",
           "--gpu-memory-utilization", "0.9", "--tensor-parallel-size", "1",
           "--attention-backend", "TRITON_ATTN", "--generation-config", "vllm",
           "--enable-auto-tool-choice", "--tool-call-parser", "hermes",
           "--no-enable-prefix-caching", "--seed", "0",
           "--host", "127.0.0.1", "--port", "8000"]
if "server" not in globals() or server.poll() is not None:
    with (ART/"server.log").open("w", encoding="utf-8") as log:
        server = subprocess.Popen(command, env=ENV, stdout=log, stderr=subprocess.STDOUT)
(ART/"server_config.json").write_text(json.dumps({"model": MODEL, "revision": REV,
    "gpu": gpu, "base_url": BASE+"/v1", "command": command}, indent=2), encoding="utf-8")
for i in range(180):
    if server.poll() is not None:
        raise RuntimeError((ART/"server.log").read_text(encoding="utf-8", errors="replace")[-6000:])
    try:
        with urllib.request.urlopen(BASE+"/v1/models", timeout=3) as response:
            assert MODEL in [item["id"] for item in json.load(response)["data"]]
            print("Server ready")
            break
    except OSError:
        time.sleep(5)
else:
    raise TimeoutError("Server startup timeout. See server.log")


In [ ]:
def complete(messages, tools=None):
    payload = {"model": MODEL, "messages": messages, "temperature": 0,
               "top_p": 1, "seed": 0, "max_tokens": 1024}
    if tools:
        payload.update(tools=tools, tool_choice="auto")
    request = urllib.request.Request(BASE+"/v1/chat/completions",
        data=json.dumps(payload).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(request, timeout=180) as response:
        return json.load(response)

# Учебный интерфейс: один читающий инструмент.
tools = [{"type":"function", "function":{"name":"get_current_day",
          "description":"Return the current date in the simulated calendar.",
          "parameters":{"type":"object","properties":{},"additionalProperties":False}}}]
messages = [{"role":"system", "content":"Use the provided calendar tool to answer the question."},
            {"role":"user", "content":"What is today's date in the simulated calendar?"}]
first = complete(messages, tools)
(ART/"tool_call.json").write_text(json.dumps(first, indent=2, ensure_ascii=False), encoding="utf-8")
message = first["choices"][0]["message"]
print(json.dumps(message, indent=2, ensure_ascii=False))
calls = message.get("tool_calls") or []
assert len(calls) == 1, "Expected one structured tool_calls entry"
assert calls[0]["function"]["name"] == "get_current_day"
assert json.loads(calls[0]["function"]["arguments"]) == {}
print("OK: structured tool_calls")


In [ ]:
# Для проверки обратного сообщения используем дату учебной среды, полученную локально.
messages.append({"role":"assistant", "content":message.get("content"), "tool_calls":calls})
messages.append({"role":"tool","tool_call_id":calls[0]["id"],"content":"2024-05-15"})
second = complete(messages)
answer = second["choices"][0]["message"].get("content")
(ART/"dialog.json").write_text(json.dumps({"messages":messages, "first":first, "second":second},
                                        indent=2, ensure_ascii=False), encoding="utf-8")
print("Answer:", answer)
print("Usage:", first.get("usage"), second.get("usage"))
assert answer and not second["choices"][0]["message"].get("tool_calls")
print("OK: tool result accepted; textual answer returned")
